In [ ]:
import torch
from ultralytics import YOLO
import cv2
import matplotlib.pyplot as plt
import imutils
import numpy as np
model = YOLO('yolov8n-obb.pt')

In [40]:
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
NVIDIA GeForce GTX 1050


In [3]:
model.to('cuda')

results= model.train(
    data='data.yaml',      # Ruta al archivo configurado arriba
    epochs=20,            # Número de vueltas al dataset
    #imgsz=640,             # Tamaño de imagen
    batch=16,              # Ajustar según la memoria de tu GPU
    name='entrenamiento_barras_obb',
    
    # Parámetros clave para Transfer Learning:
    freeze=10,             # Congela las primeras 10 capas (capas base de visión)
    lr0=0.01,              # Tasa de aprendizaje inicial
    augment=True           # Activa aumentación de datos (giros, brillo, etc.)   
)

engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=True, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=20, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=10, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1024, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n-obb.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=entrenamiento_barras_obb-2, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0, plots=True, pose=12.0, pretrained=True, profile=False, project

In [41]:
trained_model=YOLO('runs/obb/entrenamiento_barras_obb/weights/best.pt')

In [42]:
results=trained_model('test/*.jpg')



image 1/10 e:\PROYECTO FINAL - DMC\notebook\test\411.jpg: 1024x768 1 2025, 1 sbn, 44.5ms
image 2/10 e:\PROYECTO FINAL - DMC\notebook\test\412.jpg: 1024x768 1 2025, 1 sbn, 44.4ms
image 3/10 e:\PROYECTO FINAL - DMC\notebook\test\413.jpg: 1024x768 1 2025, 1 sbn, 46.0ms
image 4/10 e:\PROYECTO FINAL - DMC\notebook\test\414.jpg: 768x1024 1 2025, 1 sbn, 48.5ms
image 5/10 e:\PROYECTO FINAL - DMC\notebook\test\415.jpg: 768x1024 2 2022s, 1 2023, 1 2025, 1 sbn, 47.4ms
image 6/10 e:\PROYECTO FINAL - DMC\notebook\test\416.jpg: 768x1024 2 2020s, 1 2021, 1 2022, 2 2023s, 1 2025, 1 sbn, 43.4ms
image 7/10 e:\PROYECTO FINAL - DMC\notebook\test\417.jpg: 1024x768 1 2021, 1 2022, 1 2023, 1 2025, 2 sbns, 49.9ms
image 8/10 e:\PROYECTO FINAL - DMC\notebook\test\418.jpg: 768x1024 1 2021, 1 2022, 1 2023, 1 2025, 1 sbn, 50.3ms
image 9/10 e:\PROYECTO FINAL - DMC\notebook\test\419.jpg: 1024x768 1 2025, 1 sbn, 51.2ms
image 10/10 e:\PROYECTO FINAL - DMC\notebook\test\420.jpg: 1024x768 1 2025, 1 sbn, 44.1ms
Speed: 8

In [43]:
print (results)
print ('------------------------')
print (results[0].obb)

[ultralytics.engine.results.Results object with attributes:

boxes: None
keypoints: None
masks: None
names: {0: '2020', 1: '2021', 2: '2022', 3: '2023', 4: '2024', 5: '2025', 6: 'sbn'}
obb: ultralytics.engine.results.OBB object
orig_img: array([[[111, 142, 175],
        [116, 147, 180],
        [125, 156, 187],
        ...,
        [120, 132, 150],
        [121, 133, 151],
        [120, 132, 150]],

       [[112, 143, 176],
        [112, 143, 176],
        [117, 148, 179],
        ...,
        [122, 134, 152],
        [124, 136, 154],
        [123, 135, 153]],

       [[132, 165, 198],
        [122, 155, 188],
        [120, 151, 182],
        ...,
        [ 98, 110, 128],
        [ 97, 109, 127],
        [ 97, 109, 127]],

       ...,

       [[ 76,  61,  52],
        [ 78,  63,  54],
        [ 71,  59,  49],
        ...,
        [ 80, 105, 139],
        [ 75, 100, 134],
        [ 68,  93, 127]],

       [[ 77,  62,  53],
        [ 78,  63,  54],
        [ 70,  58,  48],
        ...,
 

In [44]:
cls_detected= results[0].obb.cls
tensor_points= results[0].obb.xyxyxyxy
print(tensor_points)

tensor([[[1834.9203, 2948.2527],
         [1824.8561, 2310.8137],
         [ 519.5589, 2331.4226],
         [ 529.6231, 2968.8616]],

        [[1857.9594, 3460.1917],
         [1847.5856, 2941.3845],
         [ 823.9129, 2961.8533],
         [ 834.2867, 3480.6604]]], device='cuda:0')


In [45]:
def crop_oriented_bbox(image, pts):
    # 1. Convertir puntos a float32 y darles forma de (4, 2)
    pts = np.array(pts, dtype="float32").reshape(4, 2)
    
    # 2. Calcular el ancho y alto del nuevo recorte
    # Distancia entre puntos (Euclidiana)
    width_a = np.sqrt(((pts[2][0] - pts[3][0]) ** 2) + ((pts[2][1] - pts[3][1]) ** 2))
    width_b = np.sqrt(((pts[1][0] - pts[0][0]) ** 2) + ((pts[1][1] - pts[0][1]) ** 2))
    max_width = max(int(width_a), int(width_b))

    height_a = np.sqrt(((pts[1][0] - pts[2][0]) ** 2) + ((pts[1][1] - pts[2][1]) ** 2))
    height_b = np.sqrt(((pts[0][0] - pts[3][0]) ** 2) + ((pts[0][1] - pts[3][1]) ** 2))
    max_height = max(int(height_a), int(height_b))

    # 3. Definir los puntos de destino para la imagen "aplanada"
    dst_pts = np.array([
        [0, 0],
        [max_width - 1, 0],
        [max_width - 1, max_height - 1],
        [0, max_height - 1]], dtype="float32")

    # 4. Calcular la matriz de transformación y aplicar el warp
    matrix = cv2.getPerspectiveTransform(pts, dst_pts)
    warped = cv2.warpPerspective(image, matrix, (max_width, max_height))
    cv2.imwrite('recorte_objeto.jpg', warped)
    #return warped

In [54]:
image=cv2.imread('test/411.jpg')
for (tensor_points_obj,cls_obj) in zip(tensor_points,cls_detected):
    wraper=crop_oriented_bbox(image,points)

In [50]:


for (tensor_points_obj,cls_obj) in zip(tensor_points,cls_detected):
    #Convertimos el tensor en numpy
    points= tensor_points_obj.cpu().numpy().squeeze()
    #Rendimensionamos y convertimos a enteros
    points= points.reshape((4,2)).astype(np.int32)
    
    print (points)
    ###########################
    cv2.drawContours(image,[points],-1,(0,255,0),2)

[[1834 2948]
 [1824 2310]
 [ 519 2331]
 [ 529 2968]]
[[1857 3460]
 [1847 2941]
 [ 823 2961]
 [ 834 3480]]


In [51]:
image= imutils.resize(image,width=1000)

In [52]:
cv2.imshow("Image",image)
cv2.waitKey(0)
cv2.destroyAllWindows()